In [ ]:
import torch
from transformers import AutoModelForTokenClassification,AutoTokenizer
from peft import PeftModel
import torch.nn.functional as F
from captum.attr import IntegratedGradients
from captum.attr import visualization as viz

In [ ]:
## NF2 example
seq="AGAGTGCAGGCCGTGGGGCGCGAGGGTCCCGGGCCTGAGCCCCGCGCCATGGCCGGGGCCATCGCTTCCCGCATGAGCTTCAGCTCTCTCAAGAGGAAGCAACCCAAGACGTTCACCGTGAGGATCGTCACCATGGACGCCGAGATGGAGTTCAATTGCGAGGTAAAGAAGCAGATTTTAGATGAAAAGATCTACTGCCCTCCTGAGGCTTCTGTGCTCCTGGCTTCTTACGCCGTCCAGGCCAAGAATAAAAAGGGCACAGAGCTGCTGCTTGGAGTGGATGCCCTGGGGCTTCACATTTATGACCCTGAGAACAGACTGACCCCCAAGATCTCCTTCCCGTGGAATGAAATCCGAAACATCTCGTACAGTGACAAGGAGTTTACTATTAAACCACTGGATAAGAAAATTGATGTCTTCAAGTTTAACTCCTCAAAGCTTCGTGTTAATAAGCTGATTCTCCAGCTATGTATCGGGAACCATGATCTATTTATGAGGAGAAGGAAAGCCGATTCTTTGGAAGTTCAGCAGATGAAAGCCCAGGCCAGGGAGGAGAAGGCTAGAAAGCAGGGCCAAAGAGGCAGATCAGCTGAAGCAGGACCTGCAGGAAGCACGCGAGGCGGAGCGAAGAGCCAAGCAGAAGCTCCTGGAGATTGCCACCAAGCCCACGTACCCGCCCATGAACCCAATTCCAGCACCGTTGCCTCCTGACATACCAAGCTTCAACCTCATTGGTGACAGCCTGTCTTTCGACTTCAAAGATACTGACATGAAGCGGCTTTCCATGGAGATAGAGAAAGAAAACTCACCTTGCAGAGCGCCAAGTCCCGAGTGGCCTTCTTTGAAGAGCTCTAGCAGGTGACCCAGCCACCCCAGGACCTGCCACTTCTCCTGCTACCGGGACCGCGGGATGGACCAGATATCAAGAGAGCCATCCATAGGGAGCTGGCTGGGGGTTTCCGTGGGAGCTCCAGAACTTTCCCCAGCTGAGTGA"

In [ ]:
# Paths
base_model_name = "InstaDeepAI/nucleotide-transformer-v2-50m-3mer-multi-species"
checkpoint_dir = "../Models/NToptimalCheckpoint"  

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)

# Load base model
base_model = AutoModelForTokenClassification.from_pretrained(
    base_model_name,
    trust_remote_code=True,
    num_labels=1
)

# Load LoRA adapters
model = PeftModel.from_pretrained(base_model, checkpoint_dir)

# Put model in eval mode
model.eval()

In [ ]:
inputs=tokenizer(seq,truncation=True)

input_ids = inputs["input_ids"]
attention_mask = torch.tensor([inputs["attention_mask"]])

tokens = tokenizer.convert_ids_to_tokens(
    input_ids
)
input_ids = torch.tensor([inputs["input_ids"]])

target_positions = [(48//3)+1,(132//3)+1] ## positions where positive TIS are present
target_class = 0

embedding_layer = model.get_input_embeddings()

def forward_embeds(inputs_embeds):

    # Captum may remove batch dim → restore it
    if inputs_embeds.dim() == 2:
        inputs_embeds = inputs_embeds.unsqueeze(0)

    outputs = model(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask
    )

    return outputs.logits

input_embeds = embedding_layer(input_ids)

ig = IntegratedGradients(forward_embeds)

In [ ]:
## NNN baseline
n_id = tokenizer.convert_tokens_to_ids("N")

baseline_ids = torch.full_like(input_ids, n_id)
baseline_embeds = embedding_layer(baseline_ids)

In [ ]:
with torch.no_grad():

    logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    ).logits
    probs=F.sigmoid(logits)

In [ ]:
records = []

for pos in target_positions:

    # ---- Integrated Gradients ----
    attr, delta = ig.attribute(
        inputs=input_embeds,
        baselines=baseline_embeds,
        target=(pos, target_class),
        n_steps=100,
        return_convergence_delta=True
    )

    # Sum embedding dim → token attribution
    attributions_sum = attr.sum(dim=-1).squeeze(0)

    # Convert to numpy
    attributions_np = attributions_sum.detach().cpu().numpy()

    # ---- Prediction metadata ----
    pred_prob = probs[0, pos, 0].item()      
    print(pred_prob)
    pred_class = int(pred_prob >= 0.5)       # convert probability to 0 or 1

    # ---- Visualization record ----
    record = viz.VisualizationDataRecord(
        word_attributions = attributions_np,
        pred_prob = pred_prob,
        pred_class = pred_class,
        true_class = 1,
        attr_class = f"Token @ position {pos}",
        attr_score = attributions_np.sum(),
        raw_input_ids = tokens,
        convergence_score = delta
    )

    records.append(record)

In [ ]:
viz.visualize_text(records)